# Customer Churn Prediction
## Group 4 – Machine Learning
### Member 1 – Logistic Regression & ML Foundation

**Objective:**  
Develop the common machine learning foundation and build a Logistic Regression churn classifier using the finalized feature-engineered dataset from Group 3.

This notebook covers:
- Dataset handoff validation
- Feature and target definition
- Train/test methodology
- Preprocessing pipeline
- Dummy baseline
- Logistic Regression
- Cross-validation
- Hyperparameter tuning
- Model evaluation
- Member 1 model handoff

In [27]:
# ==========================================
# 1. Imports and Configuration
# ==========================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42

print("Python executable   :", sys.executable)
print("Python version      :", sys.version.split()[0])
print("NumPy version       :", np.__version__)
print("Pandas version      :", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Random state        :", RANDOM_STATE)

Python executable   : d:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-Business-Intelligence-System\.venv\Scripts\python.exe
Python version      : 3.13.3
NumPy version       : 2.5.2
Pandas version      : 3.0.5
Scikit-learn version: 1.9.0
Random state        : 42


## 2. Load Group 3 Feature-Engineered Dataset

The modelling dataset is taken from the final feature-engineering output produced by Group 3.

Model preprocessing will be fitted later using training data only to avoid preprocessing leakage.

In [28]:
# ==========================================
# 2. Locate Project Root and Dataset
# ==========================================

DATA_RELATIVE_PATH = (
    Path("Feature Engineering & Statistical Analysis")
    / "outputs"
    / "customer_churn_feature_engineered.csv"
)

def find_project_root(start_path=None):
    """
    Search the current directory and its parent directories
    until the Group 3 modelling dataset is found.
    """
    current = Path(start_path or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        dataset_path = candidate / DATA_RELATIVE_PATH

        if dataset_path.exists():
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Expected to find the Group 3 feature-engineered dataset."
    )


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / DATA_RELATIVE_PATH

print("Project root:")
print(PROJECT_ROOT)

print("\nDataset:")
print(DATA_PATH)

Project root:
D:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-Business-Intelligence-System

Dataset:
D:\Internship Tasks\Group Task - Customer Churn Prediction & Business Intelligence System\Customer-Churn-Prediction-Business-Intelligence-System\Feature Engineering & Statistical Analysis\outputs\customer_churn_feature_engineered.csv


In [29]:
# ==========================================
# 3. Load Dataset
# ==========================================

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

Dataset loaded successfully.
Rows    : 7,043
Columns : 29


In [30]:
df.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Churn Value,Is_New_Customer,Tenure_Band,Is_Month_to_Month,High_Monthly_Charge,Fiber_Monthly_Risk,New_High_Spend,New_Monthly_Customer,Support_Service_Count,Security_Tech_Bundle
0,Male,0,0,0,2,1,0,DSL,1,1,...,1,1,0-6,1,0,0,0,1,2,0
1,Female,0,0,1,2,1,0,Fiber optic,0,0,...,1,1,0-6,1,1,1,1,1,0,0
2,Female,0,0,1,8,1,1,Fiber optic,0,0,...,1,1,7-12,1,1,1,1,1,1,0
3,Female,0,1,1,28,1,1,Fiber optic,0,0,...,1,0,25-48,1,1,1,0,0,2,0
4,Male,0,0,1,49,1,1,Fiber optic,0,1,...,1,0,49+,1,1,1,0,0,2,0


In [31]:
# ==========================================
# 4. Dataset Structure
# ==========================================

print("Columns:\n")

for number, column in enumerate(df.columns, start=1):
    print(f"{number:02d}. {column}")

print("\nDataset information:\n")
df.info()

Columns:

01. Gender
02. Senior Citizen
03. Partner
04. Dependents
05. Tenure Months
06. Phone Service
07. Multiple Lines
08. Internet Service
09. Online Security
10. Online Backup
11. Device Protection
12. Tech Support
13. Streaming TV
14. Streaming Movies
15. Contract
16. Paperless Billing
17. Payment Method
18. Monthly Charges
19. Total Charges
20. Churn Value
21. Is_New_Customer
22. Tenure_Band
23. Is_Month_to_Month
24. High_Monthly_Charge
25. Fiber_Monthly_Risk
26. New_High_Spend
27. New_Monthly_Customer
28. Support_Service_Count
29. Security_Tech_Bundle

Dataset information:

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Gender                 7043 non-null   str    
 1   Senior Citizen         7043 non-null   int64  
 2   Partner                7043 non-null   int64  
 3   Dependents             7043 non-null   int64  
 4 

## 3. Group 3 Data Handoff Validation

Before modelling, the dataset is checked for missing values, target integrity, required features and potential target leakage.

In [32]:
# ==========================================
# 5. Missing Value Check
# ==========================================

missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]

if missing_values.empty:
    print("No missing values detected.")
else:
    print("Missing values detected:")
    print(missing_values)

No missing values detected.


In [33]:
# ==========================================
# 6. Target Validation
# ==========================================

TARGET = "Churn Value"

if TARGET not in df.columns:
    raise ValueError(
        f"Target column '{TARGET}' was not found."
    )

print("Target:", TARGET)

print("\nUnique values:")
print(sorted(df[TARGET].dropna().unique()))

print("\nTarget counts:")
print(df[TARGET].value_counts().sort_index())

Target: Churn Value

Unique values:
[np.int64(0), np.int64(1)]

Target counts:
Churn Value
0    5174
1    1869
Name: count, dtype: int64


In [34]:
expected_target_values = {0, 1}
actual_target_values = set(df[TARGET].dropna().unique())

if actual_target_values != expected_target_values:
    raise ValueError(
        f"Unexpected target values: {actual_target_values}"
    )

print("\nTarget validation passed: binary target {0, 1}.")


Target validation passed: binary target {0, 1}.


In [35]:
# ==========================================
# 7. Leakage Protection
# ==========================================

LEAKAGE_COLUMNS = [
    "Churn Label",
    "Churn Score",
    "Churn Reason"
]

present_leakage_columns = [
    column
    for column in LEAKAGE_COLUMNS
    if column in df.columns
]

if present_leakage_columns:
    print("WARNING: Leakage-related columns exist:")
    print(present_leakage_columns)
else:
    print("No known leakage columns are present.")

No known leakage columns are present.


In [36]:
# ==========================================
# 8. Group 3 Recommended Core Features
# ==========================================

CORE_FEATURES = [
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Tenure Months",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method",
    "Monthly Charges",
    "Total Charges",
    "Fiber_Monthly_Risk",
    "New_High_Spend",
    "New_Monthly_Customer",
    "Security_Tech_Bundle"
]

print("Number of core features:", len(CORE_FEATURES))

Number of core features: 21


In [37]:
missing_core_features = [
    feature
    for feature in CORE_FEATURES
    if feature not in df.columns
]

if missing_core_features:
    raise ValueError(
        f"Missing required features: {missing_core_features}"
    )

print("All 21 Group 3 core features are available.")

All 21 Group 3 core features are available.


In [38]:
leakage_in_features = set(CORE_FEATURES).intersection(
    LEAKAGE_COLUMNS
)

if leakage_in_features:
    raise ValueError(
        f"Leakage columns detected in feature list: "
        f"{leakage_in_features}"
    )

print("Feature leakage check passed.")

Feature leakage check passed.


In [39]:
# ==========================================
# 9. Predictors and Target
# ==========================================

X = df[CORE_FEATURES].copy()
y = df[TARGET].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 21)
y shape: (7043,)


In [40]:
# ==========================================
# 10. Class Distribution
# ==========================================

class_counts = y.value_counts().sort_index()

class_percentages = (
    y.value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

class_distribution = pd.DataFrame({
    "Count": class_counts,
    "Percentage (%)": class_percentages.round(2)
})

class_distribution.index = [
    "Non-Churn (0)",
    "Churn (1)"
]

class_distribution

,Count,Percentage (%)
Non-Churn (0),5174,73.46
Churn (1),1869,26.54


### Class Distribution Interpretation

The churn target is moderately imbalanced, with non-churn customers forming the majority class.

Therefore, accuracy alone is not sufficient for model evaluation. A classifier may achieve relatively high accuracy by favouring the majority class while failing to identify customers who actually churn.

For this reason, the modelling stage will also evaluate:

- Precision
- Recall
- F1-score
- ROC-AUC

Stratification will be used during train/test splitting and cross-validation to preserve the churn distribution.